In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA = Path("/content/drive/MyDrive/HealthAISuite/Datasets/mimic-demo-1.4")

# The 8 tables this project uses (Scoped Guideline v1)
USED = {"PATIENTS", "ADMISSIONS", "ICUSTAYS", "DIAGNOSES_ICD",
        "D_ICD_DIAGNOSES", "LABEVENTS", "D_LABITEMS", "PRESCRIPTIONS"}

# 1. Everything in the folder: should be 26 CSV tables + 2 text files = 28
files = sorted(p.name for p in DATA.iterdir())
csvs = [f for f in files if f.endswith(".csv")]
print(f"{len(files)} files = {len(csvs)} CSV tables + {len(files) - len(csvs)} other files")
print("Other files:", [f for f in files if not f.endswith(".csv")])

# 2. First 5 rows of every table, with row/column counts
inventory = []
for f in csvs:
    name = f[:-4]
    head = pd.read_csv(DATA / f, nrows=5, low_memory=False)
    # Read only column 1 to count rows cheaply (CHARTEVENTS is the big one)
    n_rows = len(pd.read_csv(DATA / f, usecols=[0], low_memory=False))
    inventory.append({"table": name, "rows": n_rows, "cols": head.shape[1],
                      "used": name in USED})
    print(f"\n{'=' * 70}\n{name}  ({n_rows:,} rows x {head.shape[1]} cols)  used={name in USED}")
    display(head)

# 3. One-screen summary: used tables first
inv = pd.DataFrame(inventory).sort_values(["used", "table"], ascending=[False, True])
display(inv.reset_index(drop=True))
print("Tables used:", int(inv["used"].sum()), "of", len(inv))

In [ ]:


DATA = Path("/content/drive/MyDrive/HealthAISuite/Datasets/mimic-demo-1.4")

# The 8 in-scope tables and the job each one does (same 8 as the inventory above)
ROLE = {
    "PATIENTS":        "gender, date of birth -> age",
    "ADMISSIONS":      "one row per admission; admission details; LOS target",
    "ICUSTAYS":        "ICU stay details",
    "DIAGNOSES_ICD":   "builds the O2 labels (diabetes, heart disease); never features",
    "D_ICD_DIAGNOSES": "lookup: ICD-9 code -> description",
    "LABEVENTS":       "lab results (first 24h window, z-scored features)",
    "D_LABITEMS":      "lookup: lab itemid -> test name",
    "PRESCRIPTIONS":   "medications",
}

# Shapes seen on the first load of MIMIC-III demo v1.4. If a file is partial
# or from another version, the assert below fails instead of passing silently.
EXPECTED_SHAPES = {
    "PATIENTS": (100, 8), "ADMISSIONS": (129, 19), "ICUSTAYS": (136, 12),
    "DIAGNOSES_ICD": (1761, 5), "D_ICD_DIAGNOSES": (14567, 4),
    "LABEVENTS": (76074, 9), "D_LABITEMS": (753, 6), "PRESCRIPTIONS": (10398, 19),
}

raw = {}  # table name -> DataFrame, original values, lowercase columns
for name in ROLE:
    path = DATA / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - check the Drive mount and folder")
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.lower()  # consistent names; dates parsed later
    assert df.shape == EXPECTED_SHAPES[name], f"{name}: got {df.shape}, expected {EXPECTED_SHAPES[name]}"
    raw[name] = df

# Summary: size, exact duplicate rows, and the role of each table
summary = pd.DataFrame({
    "rows": {n: d.shape[0] for n, d in raw.items()},
    "cols": {n: d.shape[1] for n, d in raw.items()},
    "duplicate_rows": {n: int(d.duplicated().sum()) for n, d in raw.items()},
    "role": ROLE,
})
display(summary)
print(f"Loaded {len(raw)} tables | total duplicate rows: {summary['duplicate_rows'].sum()}")

## Why we use 8 of the 26 tables

The MIMIC-III demo has 26 tables (28 files with LICENSE and SHA256SUMS). We kept the smallest set that supplies the variables for O1-O3 and the business use cases, without leaking the answer.

| Objective | Business use case | Needs |
| --- | --- | --- |
| O1 Length of stay | Resource planning | ADMISSIONS (target, admission details), PATIENTS (age, gender), LABEVENTS + D_LABITEMS (first-24h labs), PRESCRIPTIONS |
| O2 Disease risk | Risk stratification (diabetes, heart disease) | DIAGNOSES_ICD + D_ICD_DIAGNOSES to build labels only; same features as O1 |
| O3 Clustering | Patient segmentation (chronic vs. acute) | Same features as O1; ICUSTAYS for ICU-stay details |

**Used (8):** PATIENTS, ADMISSIONS, ICUSTAYS, DIAGNOSES_ICD, D_ICD_DIAGNOSES, LABEVENTS, D_LABITEMS, PRESCRIPTIONS.

**Not used (18), and why**
- **NOTEEVENTS:** the demo removes every row, and no objective uses clinical notes.
- **CHARTEVENTS, DATETIMEEVENTS, INPUTEVENTS_CV/MV, OUTPUTEVENTS:** bedside time-series (vitals, fluids). No objective needs time-series, since LSTM was dropped.
- **DRGCODES:** billing groups; excluded as a leakage column per project rules.
- **PROCEDURES_ICD, PROCEDUREEVENTS_MV, CPTEVENTS:** procedure and billing records, not in the scoped variable list.
- **TRANSFERS, SERVICES, CALLOUT:** describe the stay itself (ward moves, ICU discharge clearance), so they risk leaking length of stay. Not in the scoped variable list.
- **MICROBIOLOGYEVENTS:** not in the scoped variable list.
- **D_ITEMS, D_CPT, D_ICD_PROCEDURES, CAREGIVERS:** lookups for tables we dropped.

**Rules inside the kept tables:** ICD-9 codes are assigned after discharge, so they define the O2 label and are never features. Also excluded as features: dischtime, deathtime, discharge_location, hospital_expire_flag, ADMISSIONS.diagnosis, ICUSTAYS.los.

**Caveat:** the demo patients were selected from patients who eventually die, so results are a methodology demo, not representative of all ICU patients.


## 1.3 Quality check: ADMISSIONS

ADMISSIONS is the base table (one row per `hadm_id`, 129 rows). Checks: key uniqueness, date types, nulls, LOS logic, category sizes, admissions per patient, and death-field consistency. Nothing is cleaned here, only recorded.

In [ ]:
adm = raw["ADMISSIONS"].copy()  # work on a copy; raw stays untouched

# 1. Types: parse the date columns (errors="coerce" turns bad values into NaT so we can count them)
date_cols = ["admittime", "dischtime", "deathtime", "edregtime", "edouttime"]
for c in date_cols:
    adm[c] = pd.to_datetime(adm[c], errors="coerce")

# 2. Keys: hadm_id must be unique, otherwise later merges duplicate rows
print("rows:", len(adm),
      "| unique hadm_id:", adm["hadm_id"].nunique(),
      "| unique subject_id:", adm["subject_id"].nunique())

# 3. Nulls: count and percent, only columns that have any
nulls = adm.isna().sum()
print("\nNulls:")
print(pd.DataFrame({"n": nulls, "pct": (nulls / len(adm) * 100).round(1)})[nulls > 0])

# 4. Date logic: LOS = dischtime - admittime, in days (checking the target only; dischtime is never a feature)
adm["los_days"] = (adm["dischtime"] - adm["admittime"]).dt.total_seconds() / 86400
print("\nLOS describe:\n", adm["los_days"].describe().round(2))
print("negative or zero LOS:", (adm["los_days"] <= 0).sum())

# 5. Category sizes (these become one-hot columns; many tiny categories are a problem on 129 rows)
for c in ["admission_type", "insurance", "ethnicity", "marital_status", "language", "religion"]:
    print(f"\n{c}: {adm[c].nunique()} unique")
    print(adm[c].value_counts(dropna=False).head(6))

# 6. Admissions per patient (the 15-admission outlier)
per_pt = adm.groupby("subject_id").size()
print("\nAdmissions per patient:\n", per_pt.value_counts().sort_index())
print("max:", per_pt.max(), "| share of rows from top patient:",
      round(per_pt.max() / len(adm) * 100, 1), "%")

# 7. Death fields: deathtime and hospital_expire_flag should agree (both are leakage columns)
print("\nhospital_expire_flag:", adm["hospital_expire_flag"].value_counts().to_dict())
print("deathtime present but flag=0:",
      ((adm["deathtime"].notna()) & (adm["hospital_expire_flag"] == 0)).sum())

### ADMISSIONS quality log
- **Grain:** 129 rows, 129 unique hadm_id, 100 patients. No duplicate rows.
- **Target check:** los_days computable for all 129; none <= 0. Min 0.04, median 6.63, mean 9.33, max 123.98 (right-skewed).
- **Nulls:** deathtime 89 (69.0%), matching the 89 survivors; 40 died, none with deathtime present and flag = 0. edregtime/edouttime 37 (28.7%), language 48 (37.2%), marital_status 16 (12.4%), religion 1.
- **Categories:** admission_type 92% EMERGENCY; insurance 76% Medicare; ethnicity 9 levels; marital_status has two "unknown" forms; language and religion have many tiny groups.
- **Patients:** 86 with 1 admission, 11 with 2, 2 with 3, 1 with 15 (11.6% of rows).
- **Caveats:** 31% of admissions end in in-hospital death (demo selection effect); 15-admission patient will dominate one GroupKFold fold.
- **Leakage (never features):** dischtime, deathtime, discharge_location, hospital_expire_flag, diagnosis.

In [ ]:
pat = raw["PATIENTS"].copy()

# 1. Types: parse dates (dob is shifted into the 1800s for patients older than 89)
for c in ["dob", "dod", "dod_hosp", "dod_ssn"]:
    pat[c] = pd.to_datetime(pat[c], errors="coerce")

# 2. Keys and nulls
print("rows:", len(pat), "| unique subject_id:", pat["subject_id"].nunique())
nulls = pat.isna().sum()
print("\nNulls:\n", pd.DataFrame({"n": nulls, "pct": (nulls / len(pat) * 100).round(1)})[nulls > 0])

# 3. Gender, and expire_flag (a leakage-type column, checked only)
print("\ngender:", pat["gender"].value_counts(dropna=False).to_dict())
print("expire_flag:", pat["expire_flag"].value_counts(dropna=False).to_dict())

# 4. Age at admission = year difference (safe for the shifted dob values)
a = raw["ADMISSIONS"][["hadm_id", "subject_id", "admittime"]].copy()
a["admittime"] = pd.to_datetime(a["admittime"])
age = a.merge(pat[["subject_id", "dob"]], on="subject_id", how="left")
age["age_raw"] = age["admittime"].dt.year - age["dob"].dt.year
print("\nmissing dob after merge:", age["dob"].isna().sum(), "| rows:", len(age))
print("age_raw describe:\n", age["age_raw"].describe().round(1))
print("age_raw < 0:", (age["age_raw"] < 0).sum())
print("admissions with age_raw > 89 (shifted):", (age["age_raw"] > 89).sum(),
      "| distinct patients:", age.loc[age["age_raw"] > 89, "subject_id"].nunique())
print("age_raw between 80 and 89:", age["age_raw"].between(80, 89).sum())

### PATIENTS quality log
- **Grain:** 100 rows, 100 unique subject_id. Left merge onto ADMISSIONS keeps 129 rows with 0 missing dob.
- **Gender:** F 55, M 45.
- **Nulls:** dod_hosp 30 (30%), dod_ssn 23 (23%); dod has none (every demo patient has a death date). expire_flag = 1 for all 100 (constant).
- **Age at admission (year difference):** min 17, median 73, IQR 64-83. 9 admissions (8 patients) have age_raw > 89, i.e. MIMIC-shifted values up to 300.
- **Rule for Step 2:** age = min(age_raw, 90); age_was_shifted = age_raw > 89.
- **Drop:** row_id, dob (after age), dod, dod_hosp, dod_ssn, expire_flag. Keep gender.

In [ ]:
icu = raw["ICUSTAYS"].copy()

# 1. Types
for c in ["intime", "outtime"]:
    icu[c] = pd.to_datetime(icu[c], errors="coerce")

# 2. Keys: stays vs admissions
print("rows:", len(icu), "| unique icustay_id:", icu["icustay_id"].nunique(),
      "| unique hadm_id:", icu["hadm_id"].nunique())
nulls = icu.isna().sum()
print("\nNulls:\n", pd.DataFrame({"n": nulls, "pct": (nulls / len(icu) * 100).round(1)})[nulls > 0])

# 3. Coverage against ADMISSIONS
adm_ids = set(raw["ADMISSIONS"]["hadm_id"])
print("\nICU hadm_id not in ADMISSIONS:", (~icu["hadm_id"].isin(adm_ids)).sum())
print("admissions with no ICU stay:", len(adm_ids - set(icu["hadm_id"])))
print("ICU stays per admission:\n", icu.groupby("hadm_id").size().value_counts().sort_index())

# 4. Categories
for c in ["dbsource", "first_careunit", "last_careunit"]:
    print(f"\n{c}:", icu[c].value_counts(dropna=False).to_dict())

# 5. Time logic: ICU duration, and when the first ICU stay starts relative to admission
icu["icu_hours"] = (icu["outtime"] - icu["intime"]).dt.total_seconds() / 3600
print("\nICU hours describe:\n", icu["icu_hours"].describe().round(1))
print("outtime before intime:", (icu["icu_hours"] < 0).sum())

a = raw["ADMISSIONS"][["hadm_id", "admittime"]].copy()
a["admittime"] = pd.to_datetime(a["admittime"])
first = (icu.sort_values("intime").groupby("hadm_id").first().reset_index()
            .merge(a, on="hadm_id", how="left"))
first["hrs_admit_to_icu"] = (first["intime"] - first["admittime"]).dt.total_seconds() / 3600
print("\nHours from admission to first ICU stay:\n", first["hrs_admit_to_icu"].describe().round(1))
print("ICU intime before admittime:", (first["hrs_admit_to_icu"] < 0).sum(),
      "| first ICU stay starts after 24h:", (first["hrs_admit_to_icu"] > 24).sum())

### ICUSTAYS quality log
- **Grain:** 136 stays, 136 unique icustay_id, 129 unique hadm_id. 123 admissions have 1 stay, 5 have 2, 1 has 3. No nulls in any column.
- **Coverage:** all 129 admissions have an ICU stay, so "has ICU stay" is constant and useless.
- **Care units (per stay):** MICU 77, SICU 23, CCU 19, TSICU 11, CSRU 6.
- **Timing:** ICU duration median 50.7 h (max 849.8 h), never negative. First ICU stay starts at median 0.0 h after admission; 17 admissions start after 24 h; 1 stay starts 1.7 h before admittime.
- **Rule for Step 2:** icu_by_24h flag (112 yes / 17 no) and first_careunit_24h (NONE_BY_24H when the first stay starts after 24 h).
- **Leakage (never features):** outtime, los, last_careunit, ICU hours, number of stays.

In [ ]:
dx = raw["DIAGNOSES_ICD"].copy()
dlk = raw["D_ICD_DIAGNOSES"].copy()
adm_ids = set(raw["ADMISSIONS"]["hadm_id"])

# 1. Keys, types, nulls
print("rows:", len(dx), "| admissions with >=1 code:", dx["hadm_id"].nunique(), "of", len(adm_ids))
print("icd9_code dtype:", dx["icd9_code"].dtype)
nulls = dx.isna().sum()
print("\nNulls:\n", pd.DataFrame({"n": nulls, "pct": (nulls / len(dx) * 100).round(1)})[nulls > 0])
print("duplicate (hadm_id, icd9_code):", dx.duplicated(["hadm_id", "icd9_code"]).sum())

# 2. Coverage and codes per admission
print("hadm_id not in ADMISSIONS:", (~dx["hadm_id"].isin(adm_ids)).sum())
print("admissions with no codes:", len(adm_ids - set(dx["hadm_id"])))
print("\nCodes per admission:\n", dx.groupby("hadm_id").size().describe().round(1))

# 3. Lookup table: unique codes, and do all codes have a description?
print("\nD_ICD duplicate icd9_code:", dlk["icd9_code"].duplicated().sum())
print("diagnosis codes missing from D_ICD_DIAGNOSES:", (~dx["icd9_code"].isin(dlk["icd9_code"])).sum())

# 4. Most common codes with titles (sanity check)
top = dx["icd9_code"].value_counts().head(10).rename_axis("icd9_code").reset_index(name="n")
display(top.merge(dlk[["icd9_code", "short_title"]], on="icd9_code", how="left"))

# 5. Label preview using the 3-character ICD-9 category (labels only, never features)
code = dx["icd9_code"].fillna("").astype(str).str.strip()
cat = code.str[:3]
dx["diabetes"] = cat.isin(["249", "250"])
dx["heart"] = cat.isin(["410", "411", "412", "413", "414", "428"])
print("\nDiabetes codes found:", sorted(code[dx["diabetes"]].unique()))
print("Heart codes found:", sorted(code[dx["heart"]].unique()))

# 6. Label prevalence at admission level and patient level
lab = (dx.groupby("hadm_id")[["diabetes", "heart"]].any()
         .reindex(sorted(adm_ids), fill_value=False))
print("\nAdmissions (n=%d): diabetes %d, heart %d, both %d, neither %d" % (
    len(lab), lab["diabetes"].sum(), lab["heart"].sum(),
    (lab["diabetes"] & lab["heart"]).sum(), (~lab["diabetes"] & ~lab["heart"]).sum()))
pt = lab.join(raw["ADMISSIONS"].set_index("hadm_id")["subject_id"]).groupby("subject_id").any()
print("Patients (n=%d): diabetes %d, heart %d" % (len(pt), pt["diabetes"].sum(), pt["heart"].sum()))

In [ ]:
missing = dx.loc[~dx["icd9_code"].isin(dlk["icd9_code"]), "icd9_code"]
print(missing.value_counts().head(15))
print("label codes among them:", sorted(set(missing) & set(code[dx["diabetes"] | dx["heart"]])))

### DIAGNOSES_ICD quality log
- **Coverage:** 1,761 rows; all 129 admissions have >= 1 code (min 3, median 12, max 37). No nulls, no duplicate (hadm_id, icd9_code).
- **Lookup:** D_ICD_DIAGNOSES has unique codes; 45 diagnosis rows (2.6%) have no title there. Labels use the code prefix, so they are unaffected.
- **Labels (3-character ICD-9 category):** diabetes 249/250, heart disease 410-414 and 428.
- **Admissions (n=129):** diabetes 48 (37.2%), heart 52 (40.3%), both 23, neither 52. **Patients (n=100):** diabetes 31, heart 47.
- **Caveats:** about 6 diabetes-positive patients per test fold; "heart disease" excludes atrial fibrillation and hypertension.
- **Never features:** any ICD code, flag or count derived from DIAGNOSES_ICD (labels only).

- **Missing titles:** the 45 rows without a lookup title are non-label codes (e.g. 0414, 2765, 5185); checked against the label code lists, none overlap.
- **Rule for Step 2:** read icd9_code as string (codes like 0414 have a leading zero; numeric parsing would turn it into 414 and mislabel heart disease).

In [ ]:
lab = raw["LABEVENTS"].copy()
items = raw["D_LABITEMS"].copy()
lab["charttime"] = pd.to_datetime(lab["charttime"], errors="coerce")

# 1. Nulls and lookup coverage
nulls = lab.isna().sum()
print("Nulls:\n", pd.DataFrame({"n": nulls, "pct": (nulls / len(lab) * 100).round(1)})[nulls > 0])
print("distinct itemid:", lab["itemid"].nunique(),
      "| itemid missing from D_LABITEMS:", (~lab["itemid"].isin(items["itemid"])).sum(),
      "| duplicate itemid in D_LABITEMS:", items["itemid"].duplicated().sum())

# 2. Rows with no hadm_id are outpatient labs: they cannot be linked to an admission
no_h = lab["hadm_id"].isna()
print("\nrows without hadm_id:", no_h.sum(), f"({no_h.mean() * 100:.1f}%)")
lab = lab[~no_h].copy()
lab["hadm_id"] = lab["hadm_id"].astype("int64")

# 3. Link to admission time and compute hours since admission
adm = raw["ADMISSIONS"][["hadm_id", "admittime"]].copy()
adm["admittime"] = pd.to_datetime(adm["admittime"])
n_before = len(lab)
lab = lab.merge(adm, on="hadm_id", how="inner")
lab["hrs"] = (lab["charttime"] - lab["admittime"]).dt.total_seconds() / 3600
print("linked rows:", len(lab), "of", n_before, "| admissions with any lab:", lab["hadm_id"].nunique())
print("hours since admission:\n", lab["hrs"].describe().round(1))

# 4. Window comparison: how many admissions does each window cover?
for lo, hi in [(0, 24), (-6, 24), (-24, 24)]:
    w = lab[(lab["hrs"] >= lo) & (lab["hrs"] <= hi)]
    print(f"window {lo:>4}h to {hi}h: rows={len(w):>6} | admissions covered={w['hadm_id'].nunique()} of {len(adm)}"
          f" | distinct items={w['itemid'].nunique()}")

# 5. Item coverage in the -6h to 24h window: which tests are measured often enough to use?
W = lab[(lab["hrs"] >= -6) & (lab["hrs"] <= 24)]
cov = (W.groupby("itemid")["hadm_id"].nunique().rename("n_adm").reset_index()
         .merge(items[["itemid", "label", "fluid", "category"]], on="itemid", how="left")
         .sort_values("n_adm", ascending=False))
cov["pct_adm"] = (cov["n_adm"] / len(adm) * 100).round(1)
display(cov.head(25))
print("items in >=50% of admissions:", (cov["pct_adm"] >= 50).sum(),
      "| >=80%:", (cov["pct_adm"] >= 80).sum())

# 6. Repeats per admission-item, and non-numeric values (decides how to aggregate)
print("\nmeasurements per admission-item:\n", W.groupby(["hadm_id", "itemid"]).size().describe().round(1))
print("valuenum missing in window:", round(W["valuenum"].isna().mean() * 100, 1), "%")

### LABEVENTS quality log
- **Linking:** 76,074 rows; 14,262 (18.7%) have no hadm_id and are dropped. The remaining 61,812 all link to ADMISSIONS (129 admissions have labs). 439 items, all present in D_LABITEMS.
- **Missing:** valuenum 11.9%, valueuom 12.4%, flag 60.9% (flag is only set for abnormal results). Use valuenum.
- **Timing:** hours since admission range from -12.2 to 2,960.9 (median 107.1).
- **Window decision:** 0-24h covers 128/129 admissions (9,673 rows); -6h to 24h covers 129/129 (14,273 rows); -24h adds only 572 rows. Chosen: -6h to 24h (still known by hour 24).
- **Items:** 273 items in the window; 59 in >= 50% of admissions, 23 in >= 80%. Use the 23 with >= 80% as the candidate pool, then prune correlated items. Recompute coverage after dropping non-numeric results (17.6% of window rows).
- **Aggregation:** one value per item = mean in window (median 1 measurement per admission-item, max 16).
- **Caveat:** glucose is close to defining the diabetes label.

In [ ]:
rx = raw["PRESCRIPTIONS"].copy()
for c in ["startdate", "enddate"]:
    rx[c] = pd.to_datetime(rx[c], errors="coerce")

# 1. Keys, nulls, and whether dates carry a time of day
print("rows:", len(rx), "| unique hadm_id:", rx["hadm_id"].nunique(),
      "| hadm_id null:", rx["hadm_id"].isna().sum())
nulls = rx.isna().sum()
print("\nNulls:\n", pd.DataFrame({"n": nulls, "pct": (nulls / len(rx) * 100).round(1)})[nulls > 0])
print("startdate is date-only (midnight):", (rx["startdate"].dt.normalize() == rx["startdate"]).mean())

# 2. Categories
print("\ndrug_type:", rx["drug_type"].value_counts(dropna=False).to_dict())
print("distinct drugs:", rx["drug"].nunique())
print("top routes:", rx["route"].value_counts(dropna=False).head(8).to_dict())
display(rx["drug"].value_counts().head(15).rename_axis("drug").reset_index(name="n"))

# 3. Start day relative to the admission DATE (startdate has no time of day)
adm = raw["ADMISSIONS"][["hadm_id", "admittime"]].copy()
adm["admittime"] = pd.to_datetime(adm["admittime"])
rx = rx.dropna(subset=["hadm_id"]).copy()
rx["hadm_id"] = rx["hadm_id"].astype("int64")
rx = rx.merge(adm, on="hadm_id", how="inner")
rx["day_offset"] = (rx["startdate"] - rx["admittime"].dt.normalize()).dt.days
print("\nday_offset describe:\n", rx["day_offset"].describe().round(1))
print("started before admission date:", (rx["day_offset"] < 0).sum())

# 4. Two ways to approximate "first 24h": strict (admission date only) vs. admission date + next day
for label, cut in [("strict: day_offset <= 0", 0), ("wider: day_offset <= 1", 1)]:
    w = rx[rx["day_offset"] <= cut]
    print(f"{label}: rows={len(w)} | admissions covered={w['hadm_id'].nunique()} of {len(adm)}")

# 5. Drugs that nearly define the labels (candidates to exclude; confirm with Bala)
W = rx[rx["day_offset"] <= 1]
d = W["drug"].fillna("").str.lower()
for name, pat in {"diabetes drugs": "insulin|metformin|glipizide|glyburide|glimepiride|pioglitazone|sitagliptin",
                  "heart drugs": "nitroglycerin|isosorbide|clopidogrel|digoxin|carvedilol"}.items():
    m = d.str.contains(pat)
    print(f"{name}: rows={m.sum()} | admissions={W.loc[m, 'hadm_id'].nunique()}")

### PRESCRIPTIONS quality log
- **Coverage:** 10,398 rows; 122 of 129 admissions have any prescription (7 get count 0). No null hadm_id.
- **Dates:** startdate is date-only (all midnight), so the 24h window is approximated by calendar day.
- **Windows:** strict (start date <= admission date) covers 104/129 admissions (2,264 rows); wider (<= admission date + 1 day) covers 122/129 (4,346 rows) but can include orders up to ~48h after admission. Chosen: strict; wider kept as a sensitivity check.
- **Columns:** drug has no nulls; drug_name_generic and drug_name_poe are 44.5% null; icustay_id 32.2% null (non-ICU orders). drug_type: MAIN 7,709, BASE 2,617, ADDITIVE 72.
- **Names:** 592 distinct drug strings with many spelling variants (D5W / 5% Dextrose), so drug-level features are not used.
- **Features for Step 2:** n_orders_24h and n_unique_drugs_24h from MAIN orders only, excluding label-defining drugs.
- **Label-defining drugs (draft keyword list, to confirm with Bala):** diabetes drugs matched 154 rows / 80 admissions (vs 48 diabetes labels); heart drugs 42 rows / 26 admissions.

## Step 1 summary: decisions carried into Step 2

| Table | Build / keep | Drop |
| --- | --- | --- |
| ADMISSIONS | los_days (target); admission_type, insurance, ethnicity, marital_status (grouped); via_ed candidate | religion, language, dischtime, deathtime, discharge_location, hospital_expire_flag, diagnosis, ED times, row_id |
| PATIENTS | gender; age = min(year difference, 90); age_was_shifted | row_id, dob, dod, dod_hosp, dod_ssn, expire_flag |
| ICUSTAYS | icu_by_24h; first_careunit_24h | outtime, los, last_careunit, ward ids, number of stays, ICU hours, dbsource |
| DIAGNOSES_ICD | diabetes and heart_disease labels only | every code, flag or count as a feature |
| LABEVENTS | window -6h to 24h; items in >= 80% of admissions (recount after numeric filter); mean per item; prune correlated items | rows without hadm_id; non-numeric results |
| PRESCRIPTIONS | strict window; n_orders_24h, n_unique_drugs_24h (MAIN only, label-defining drugs excluded) | drug-level columns; label-defining drugs |

**Open for Bala:** lab window (-6h vs 0h); prescription window (strict vs wider); heart-disease definition (excludes atrial fibrillation, hypertension); O2 as two binary flags or one 4-class label; comorbidity flags (guideline vs CLAUDE.md).

In [ ]:
# --- Grouping rules (edit here if you or Bala disagree) ---
ADMISSION_TYPE_MAP = {"EMERGENCY": "EMERGENCY", "ELECTIVE": "NON_EMERGENCY", "URGENT": "NON_EMERGENCY"}
INSURANCE_MAP = {"Medicare": "MEDICARE", "Private": "PRIVATE", "Medicaid": "MEDICAID_GOV", "Government": "MEDICAID_GOV"}

def group_ethnicity(s):
    """Keyword rules so unseen levels are handled; everything else -> OTHER_UNKNOWN."""
    s = str(s).upper()
    for key, grp in [("WHITE", "WHITE"), ("HISPANIC", "HISPANIC"), ("LATINO", "HISPANIC"),
                     ("BLACK", "BLACK"), ("ASIAN", "ASIAN")]:
        if key in s:
            return grp
    return "OTHER_UNKNOWN"

adm = raw["ADMISSIONS"].copy()
pat = raw["PATIENTS"].copy()
for c in ["admittime", "dischtime", "edregtime"]:
    adm[c] = pd.to_datetime(adm[c])
pat["dob"] = pd.to_datetime(pat["dob"])

# Target: hospital length of stay in days (NOT ICUSTAYS.los)
adm["los_days"] = (adm["dischtime"] - adm["admittime"]).dt.total_seconds() / 86400

# Grouped categories (marital: NaN and "UNKNOWN (DEFAULT)" both -> UNKNOWN)
adm["admission_type_grp"] = adm["admission_type"].map(ADMISSION_TYPE_MAP)
adm["insurance_grp"] = adm["insurance"].map(INSURANCE_MAP)
adm["ethnicity_grp"] = adm["ethnicity"].map(group_ethnicity)
adm["marital_grp"] = adm["marital_status"].replace({"UNKNOWN (DEFAULT)": None}).fillna("UNKNOWN")
adm["via_ed"] = adm["edregtime"].notna().astype(int)   # candidate feature; EDA decides

# Age at admission: year difference (safe for MIMIC's shifted dob), capped at 90
adm = adm.merge(pat[["subject_id", "gender", "dob"]], on="subject_id", how="left")
age_raw = adm["admittime"].dt.year - adm["dob"].dt.year
adm["age_was_shifted"] = (age_raw > 89).astype(int)
adm["age"] = age_raw.clip(upper=90)

# Base table: one row per admission. subject_id is kept only for GroupKFold.
base = adm[["hadm_id", "subject_id", "los_days", "age", "age_was_shifted", "gender",
            "admission_type_grp", "insurance_grp", "ethnicity_grp", "marital_grp", "via_ed"]].copy()

# --- Checks ---
assert len(base) == 129 and base["hadm_id"].is_unique, "grain broken"
assert base[["los_days", "age", "gender"]].notna().all().all(), "unexpected nulls"
print("base:", base.shape, "| shifted ages:", base["age_was_shifted"].sum(),
      "| age max:", base["age"].max())
print("\nEthnicity mapping (original -> group):")
display(adm.groupby(["ethnicity", "ethnicity_grp"]).size().rename("n").reset_index())
for c in ["admission_type_grp", "insurance_grp", "ethnicity_grp", "marital_grp", "via_ed"]:
    print(f"\n{c}:", base[c].value_counts(dropna=False).to_dict())
print("\nvia_ed vs admission_type:")
print(pd.crosstab(adm["admission_type"], adm["via_ed"]))
print("\nadmission_location:", adm["admission_location"].value_counts().to_dict())

In [ ]:
import numpy as np

# --- 2b-1: fixes after reviewing the output ---
base["marital_grp"] = base["marital_grp"].replace({"SEPARATED": "DIVORCED_SEPARATED",
                                                   "DIVORCED": "DIVORCED_SEPARATED"})
LOCATION_MAP = {"EMERGENCY ROOM ADMIT": "ER",
                "TRANSFER FROM HOSP/EXTRAM": "TRANSFER_IN", "TRANSFER FROM SKILLED NUR": "TRANSFER_IN",
                "CLINIC REFERRAL/PREMATURE": "REFERRAL", "PHYS REFERRAL/NORMAL DELI": "REFERRAL"}
loc = adm.set_index("hadm_id")["admission_location"].map(LOCATION_MAP)
base["admission_location_grp"] = base["hadm_id"].map(loc)
base = base.drop(columns="via_ed")          # redundant with admission_type + location
assert base["admission_location_grp"].notna().all(), "unmapped admission_location"

# --- 2b-2: ICU features, only what is known by hour 24 ---
icu = raw["ICUSTAYS"].copy()
icu["intime"] = pd.to_datetime(icu["intime"])
first = (icu.sort_values("intime").groupby("hadm_id").first().reset_index()   # first stay per admission
            .merge(adm[["hadm_id", "admittime"]], on="hadm_id", how="left"))
hrs = (first["intime"] - first["admittime"]).dt.total_seconds() / 3600
first["icu_by_24h"] = (hrs.clip(lower=0) <= 24).astype(int)                    # clip the -1.7h case to 0
first["first_careunit_24h"] = np.where(first["icu_by_24h"] == 1,
                                       first["first_careunit"], "NONE_BY_24H")
base = base.merge(first[["hadm_id", "icu_by_24h", "first_careunit_24h"]], on="hadm_id", how="left")

# --- 2b-3: O2 labels from ICD-9 (labels only, never features) ---
dx = raw["DIAGNOSES_ICD"].copy()
dx["icd9_code"] = dx["icd9_code"].astype(str).str.strip()   # stays text: '0414' must not become 414
cat3 = dx["icd9_code"].str[:3]
dx["diabetes"] = cat3.isin(["249", "250"])
dx["heart_disease"] = cat3.isin(["410", "411", "412", "413", "414", "428"])
labels = dx.groupby("hadm_id")[["diabetes", "heart_disease"]].any().astype(int).reset_index()
base = base.merge(labels, on="hadm_id", how="left")
base[["diabetes", "heart_disease"]] = base[["diabetes", "heart_disease"]].fillna(0).astype(int)

# --- Checks: grain, nulls, and agreement with the Step 1 numbers ---
assert len(base) == 129 and base["hadm_id"].is_unique
assert base.drop(columns=[]).notna().all().all(), "unexpected nulls"
assert base["diabetes"].sum() == 48 and base["heart_disease"].sum() == 52, "label counts differ from Step 1"
print("base:", base.shape)
print("columns:", list(base.columns))
print("icu_by_24h:", base["icu_by_24h"].value_counts().to_dict())
print("first_careunit_24h:", base["first_careunit_24h"].value_counts().to_dict())
print("admission_location_grp:", base["admission_location_grp"].value_counts().to_dict())
print("marital_grp:", base["marital_grp"].value_counts().to_dict())

In [ ]:
import re

lab = raw["LABEVENTS"].copy()
lab["charttime"] = pd.to_datetime(lab["charttime"])
lab = lab.dropna(subset=["hadm_id", "valuenum"]).copy()    # drop outpatient (no hadm_id) and non-numeric results
lab["hadm_id"] = lab["hadm_id"].astype("int64")
lab = lab.merge(adm[["hadm_id", "admittime"]], on="hadm_id", how="inner")
lab["hrs"] = (lab["charttime"] - lab["admittime"]).dt.total_seconds() / 3600
W = lab[(lab["hrs"] >= -6) & (lab["hrs"] <= 24)].copy()    # window decided in Step 1
names = raw["D_LABITEMS"].set_index("itemid")["label"]

# --- Units: normalise case/spaces, map spelling variants, then handle any genuine mix ---
UNIT_ALIASES = {"seconds": "sec", "uu/ml": "uiu/ml"}        # same unit, different spelling (seen in the data)
W["unit"] = W["valueuom"].str.strip().str.lower().replace(UNIT_ALIASES)   # missing unit stays NaN
n_units = W.groupby("itemid")["unit"].nunique()
mixed = n_units[n_units > 1].index
print("items still with >1 unit:", [(i, names[i]) for i in mixed])
dominant = W[W["itemid"].isin(mixed)].groupby("itemid")["unit"].agg(lambda s: s.value_counts().idxmax())
keep_row = ~W["itemid"].isin(mixed) | (W["unit"] == W["itemid"].map(dominant))
print("rows dropped for non-dominant units, by item:")
print(W.loc[~keep_row, "itemid"].map(names).value_counts())
W = W[keep_row]

# --- Coverage recomputed AFTER the numeric and unit filters; keep items in >= 80% of admissions ---
n_adm = base["hadm_id"].nunique()
cov = W.groupby("itemid")["hadm_id"].nunique() / n_adm
keep = cov[cov >= 0.80].index
assert len(keep) > 0, "no lab items kept"
assert not set(keep) & set(mixed), "a kept lab still has mixed units - review before using it"
print("items kept (numeric, single unit, >=80% of admissions):", len(keep))

# --- One value per admission-item: mean in window. Missing stays NaN (imputed later in the Pipeline) ---
wide = W[W["itemid"].isin(keep)].groupby(["hadm_id", "itemid"])["valuenum"].mean().unstack()
wide.columns = [f"lab_{i}_{re.sub(r'[^a-z0-9]+', '_', names[i].lower()).strip('_')}" for i in wide.columns]
labs_wide = wide.reindex(base["hadm_id"]).rename_axis("hadm_id").reset_index()

# --- Checks: missingness and plausible ranges ---
feat = labs_wide.drop(columns="hadm_id")
print("rows:", len(labs_wide), "| lab columns:", feat.shape[1],
      "| worst missing %:", round(feat.isna().mean().max() * 100, 1))
print("PT admissions with a value:", int(feat["lab_51274_pt"].notna().sum()))
display(feat.describe().T[["count", "min", "50%", "max"]].round(1))
print("columns with min <= 0 (check these):", feat.columns[feat.min() <= 0].tolist())

In [ ]:
# --- Prescriptions: strict window (start date <= admission date), MAIN orders only ---
rx = raw["PRESCRIPTIONS"].copy()
rx["startdate"] = pd.to_datetime(rx["startdate"])
rx = rx.merge(adm[["hadm_id", "admittime"]], on="hadm_id", how="inner")
rx["day_offset"] = (rx["startdate"] - rx["admittime"].dt.normalize()).dt.days
E = rx[(rx["day_offset"] <= 0) & (rx["drug_type"] == "MAIN")].copy()

# Drop label-defining drugs (draft list, to confirm with Bala)
LABEL_DRUGS = ("insulin|metformin|glipizide|glyburide|glimepiride|pioglitazone|sitagliptin"
               "|nitroglycerin|isosorbide|clopidogrel|digoxin|carvedilol")
E["drug_norm"] = E["drug"].fillna("").str.lower().str.strip()
E = E[~E["drug_norm"].str.contains(LABEL_DRUGS)]

rx_feat = (E.groupby("hadm_id").agg(n_orders_24h=("drug", "size"),
                                    n_unique_drugs_24h=("drug_norm", "nunique"))
             .reindex(pd.Index(base["hadm_id"], name="hadm_id"), fill_value=0)   # no orders -> 0 (assumption)
             .reset_index())
print("admissions with 0 orders in window:", int((rx_feat["n_orders_24h"] == 0).sum()))
print(rx_feat[["n_orders_24h", "n_unique_drugs_24h"]].describe().round(1))

# --- Master table: one row per admission ---
master = (base.merge(labs_wide, on="hadm_id", how="left")
              .merge(rx_feat, on="hadm_id", how="left"))
assert len(master) == 129 and master["hadm_id"].is_unique, "grain broken"

non_lab = [c for c in master.columns if not c.startswith("lab_")]
assert master[non_lab].notna().all().all(), "unexpected nulls outside labs"
FORBIDDEN = {"dischtime", "deathtime", "discharge_location", "hospital_expire_flag", "diagnosis", "los"}
assert not FORBIDDEN & set(master.columns), "leakage column present"
assert master["diabetes"].sum() == 48 and master["heart_disease"].sum() == 52, "labels changed"
n_lab = sum(c.startswith("lab_") for c in master.columns)
print("master:", master.shape, "| lab columns:", n_lab,
      "| lab missing share:", round(master.filter(like="lab_").isna().mean().mean() * 100, 1), "%")

# Save outside the repo (Drive) so a restart does not cost the work. Never commit this file.
from pathlib import Path
OUT = Path("/content/drive/MyDrive/HealthAISuite/Processed")
OUT.mkdir(parents=True, exist_ok=True)
master.to_csv(OUT / "master.csv", index=False)
print("saved:", OUT / "master.csv")

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

# ================= CONFIG: every Step 1 decision lives here =================
DATA = Path("/content/drive/MyDrive/HealthAISuite/Datasets/mimic-demo-1.4")
OUT = Path("/content/drive/MyDrive/HealthAISuite/Processed")

TABLES = ["PATIENTS", "ADMISSIONS", "ICUSTAYS", "DIAGNOSES_ICD",
          "D_ICD_DIAGNOSES", "LABEVENTS", "D_LABITEMS", "PRESCRIPTIONS"]
EXPECTED_SHAPES = {"PATIENTS": (100, 8), "ADMISSIONS": (129, 19), "ICUSTAYS": (136, 12),
                   "DIAGNOSES_ICD": (1761, 5), "D_ICD_DIAGNOSES": (14567, 4),
                   "LABEVENTS": (76074, 9), "D_LABITEMS": (753, 6), "PRESCRIPTIONS": (10398, 19)}
STR_COLS = {"DIAGNOSES_ICD": {"ICD9_CODE": str}, "D_ICD_DIAGNOSES": {"ICD9_CODE": str}}  # keep leading zeros

AGE_CAP = 90                                   # MIMIC shifts ages > 89 to ~300
ICU_WINDOW_H = 24
LAB_WINDOW_H = (-6, 24)                        # hours from admission
LAB_MIN_COVERAGE = 0.80
UNIT_ALIASES = {"seconds": "sec", "uu/ml": "uiu/ml"}   # same unit, different spelling

ADMISSION_TYPE_MAP = {"EMERGENCY": "EMERGENCY", "ELECTIVE": "NON_EMERGENCY", "URGENT": "NON_EMERGENCY"}
INSURANCE_MAP = {"Medicare": "MEDICARE", "Private": "PRIVATE", "Medicaid": "MEDICAID_GOV", "Government": "MEDICAID_GOV"}
LOCATION_MAP = {"EMERGENCY ROOM ADMIT": "ER",
                "TRANSFER FROM HOSP/EXTRAM": "TRANSFER_IN", "TRANSFER FROM SKILLED NUR": "TRANSFER_IN",
                "CLINIC REFERRAL/PREMATURE": "REFERRAL", "PHYS REFERRAL/NORMAL DELI": "REFERRAL"}
DIABETES_CATS = ["249", "250"]                              # ICD-9 3-character categories (label only)
HEART_CATS = ["410", "411", "412", "413", "414", "428"]
LABEL_DRUGS = ("insulin|metformin|glipizide|glyburide|glimepiride|pioglitazone|sitagliptin"
               "|nitroglycerin|isosorbide|clopidogrel|digoxin|carvedilol")   # draft list, confirm with Bala
FORBIDDEN = {"dischtime", "deathtime", "discharge_location", "hospital_expire_flag", "diagnosis", "los"}
EXPECTED = {"rows": 129, "diabetes": 48, "heart_disease": 52, "icu_by_24h": 112}   # Step 1 numbers


# ================= Functions: one per table =================
def load_tables():
    """Load the 8 in-scope tables with lowercase columns and check their shapes."""
    raw = {}
    for name in TABLES:
        df = pd.read_csv(DATA / f"{name}.csv", low_memory=False, dtype=STR_COLS.get(name))
        df.columns = df.columns.str.lower()
        assert df.shape == EXPECTED_SHAPES[name], f"{name}: got {df.shape}"
        raw[name] = df
    return raw


def group_ethnicity(s):
    """Keyword rules; anything else -> OTHER_UNKNOWN."""
    s = str(s).upper()
    for key, grp in [("WHITE", "WHITE"), ("HISPANIC", "HISPANIC"), ("LATINO", "HISPANIC"),
                     ("BLACK", "BLACK"), ("ASIAN", "ASIAN")]:
        if key in s:
            return grp
    return "OTHER_UNKNOWN"


def build_base(raw):
    """ADMISSIONS + PATIENTS -> one row per admission: target los_days, age, grouped categories."""
    adm, pat = raw["ADMISSIONS"].copy(), raw["PATIENTS"].copy()
    for c in ["admittime", "dischtime"]:
        adm[c] = pd.to_datetime(adm[c])
    pat["dob"] = pd.to_datetime(pat["dob"])
    adm["los_days"] = (adm["dischtime"] - adm["admittime"]).dt.total_seconds() / 86400   # NOT ICUSTAYS.los
    adm["admission_type_grp"] = adm["admission_type"].map(ADMISSION_TYPE_MAP)
    adm["insurance_grp"] = adm["insurance"].map(INSURANCE_MAP)
    adm["ethnicity_grp"] = adm["ethnicity"].map(group_ethnicity)
    adm["marital_grp"] = (adm["marital_status"].replace({"UNKNOWN (DEFAULT)": None}).fillna("UNKNOWN")
                          .replace({"SEPARATED": "DIVORCED_SEPARATED", "DIVORCED": "DIVORCED_SEPARATED"}))
    adm["admission_location_grp"] = adm["admission_location"].map(LOCATION_MAP)
    adm = adm.merge(pat[["subject_id", "gender", "dob"]], on="subject_id", how="left")
    age_raw = adm["admittime"].dt.year - adm["dob"].dt.year        # year difference: safe for shifted dob
    adm["age_was_shifted"] = (age_raw > 89).astype(int)
    adm["age"] = age_raw.clip(upper=AGE_CAP)
    cols = ["hadm_id", "subject_id", "los_days", "age", "age_was_shifted", "gender", "admission_type_grp",
            "insurance_grp", "ethnicity_grp", "marital_grp", "admission_location_grp"]
    return adm[cols].copy(), adm[["hadm_id", "admittime"]].copy()


def add_icu_features(base, raw, adm_times):
    """First ICU stay per admission; only what is known by hour 24."""
    icu = raw["ICUSTAYS"].copy()
    icu["intime"] = pd.to_datetime(icu["intime"])
    first = (icu.sort_values("intime").groupby("hadm_id").first().reset_index()
                .merge(adm_times, on="hadm_id", how="left"))
    hrs = (first["intime"] - first["admittime"]).dt.total_seconds() / 3600
    first["icu_by_24h"] = (hrs.clip(lower=0) <= ICU_WINDOW_H).astype(int)
    first["first_careunit_24h"] = np.where(first["icu_by_24h"] == 1, first["first_careunit"], "NONE_BY_24H")
    return base.merge(first[["hadm_id", "icu_by_24h", "first_careunit_24h"]], on="hadm_id", how="left")


def add_labels(base, raw):
    """O2 labels from ICD-9 category prefixes. Labels only, never features."""
    dx = raw["DIAGNOSES_ICD"].copy()
    cat3 = dx["icd9_code"].astype(str).str.strip().str[:3]
    dx["diabetes"], dx["heart_disease"] = cat3.isin(DIABETES_CATS), cat3.isin(HEART_CATS)
    labels = dx.groupby("hadm_id")[["diabetes", "heart_disease"]].any().astype(int).reset_index()
    out = base.merge(labels, on="hadm_id", how="left")
    out[["diabetes", "heart_disease"]] = out[["diabetes", "heart_disease"]].fillna(0).astype(int)
    return out


def build_lab_features(base, raw, adm_times):
    """Mean per item in the lab window; numeric, single-unit items present in >= 80% of admissions."""
    lab = raw["LABEVENTS"].copy()
    lab["charttime"] = pd.to_datetime(lab["charttime"])
    lab = lab.dropna(subset=["hadm_id", "valuenum"]).copy()          # outpatient and non-numeric rows
    lab["hadm_id"] = lab["hadm_id"].astype("int64")
    lab = lab.merge(adm_times, on="hadm_id", how="inner")
    lab["hrs"] = (lab["charttime"] - lab["admittime"]).dt.total_seconds() / 3600
    W = lab[(lab["hrs"] >= LAB_WINDOW_H[0]) & (lab["hrs"] <= LAB_WINDOW_H[1])].copy()
    names = raw["D_LABITEMS"].set_index("itemid")["label"]

    W["unit"] = W["valueuom"].str.strip().str.lower().replace(UNIT_ALIASES)
    n_units = W.groupby("itemid")["unit"].nunique()
    mixed = n_units[n_units > 1].index                               # genuinely different units
    dominant = W[W["itemid"].isin(mixed)].groupby("itemid")["unit"].agg(lambda s: s.value_counts().idxmax())
    W = W[~W["itemid"].isin(mixed) | (W["unit"] == W["itemid"].map(dominant))]

    cov = W.groupby("itemid")["hadm_id"].nunique() / base["hadm_id"].nunique()
    keep = cov[cov >= LAB_MIN_COVERAGE].index
    assert len(keep) > 0 and not set(keep) & set(mixed), "lab selection problem"
    wide = W[W["itemid"].isin(keep)].groupby(["hadm_id", "itemid"])["valuenum"].mean().unstack()
    wide.columns = [f"lab_{i}_{re.sub(r'[^a-z0-9]+', '_', names[i].lower()).strip('_')}" for i in wide.columns]
    return wide.reindex(base["hadm_id"]).rename_axis("hadm_id").reset_index()   # missing stays NaN


def build_rx_features(base, raw, adm_times):
    """Counts of MAIN orders on/before the admission date, excluding label-defining drugs. None -> 0."""
    rx = raw["PRESCRIPTIONS"].copy()
    rx["startdate"] = pd.to_datetime(rx["startdate"])
    rx = rx.merge(adm_times, on="hadm_id", how="inner")
    rx["day_offset"] = (rx["startdate"] - rx["admittime"].dt.normalize()).dt.days
    E = rx[(rx["day_offset"] <= 0) & (rx["drug_type"] == "MAIN")].copy()
    E["drug_norm"] = E["drug"].fillna("").str.lower().str.strip()
    E = E[~E["drug_norm"].str.contains(LABEL_DRUGS)]
    return (E.groupby("hadm_id").agg(n_orders_24h=("drug", "size"), n_unique_drugs_24h=("drug_norm", "nunique"))
              .reindex(pd.Index(base["hadm_id"], name="hadm_id"), fill_value=0).reset_index())


def build_master(raw):
    """Run every step and check the result."""
    base, adm_times = build_base(raw)
    base = add_labels(add_icu_features(base, raw, adm_times), raw)
    master = (base.merge(build_lab_features(base, raw, adm_times), on="hadm_id", how="left")
                  .merge(build_rx_features(base, raw, adm_times), on="hadm_id", how="left"))
    assert len(master) == EXPECTED["rows"] and master["hadm_id"].is_unique, "grain broken"
    assert master[[c for c in master.columns if not c.startswith("lab_")]].notna().all().all(), "nulls outside labs"
    assert not FORBIDDEN & set(master.columns), "leakage column present"
    assert master["diabetes"].sum() == EXPECTED["diabetes"] and master["heart_disease"].sum() == EXPECTED["heart_disease"]
    assert master["icu_by_24h"].sum() == EXPECTED["icu_by_24h"]
    return master


# ================= Run, and prove it reproduces the step-by-step master =================
raw = load_tables()
master_new = build_master(raw)
old = pd.read_csv(OUT / "master.csv")
pd.testing.assert_frame_equal(master_new.reset_index(drop=True), old, check_dtype=False)
print("master rebuilt:", master_new.shape, "| identical to the saved step-by-step master")

## 4. EDA on the master table

Goal: understand the data, not select features. Nothing here fits a model. Feature selection that uses the target happens inside cross-validation in Step 5.

### 4a. Targets
LOS (O1) shape and outliers; label balance (O2); the patient with the most admissions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Setup: make sure master and adm exist (works even after a restart)
master = master_new
adm = raw["ADMISSIONS"].copy()    # used only for analysis-only death checks, never as features
adm["los_days"] = (pd.to_datetime(adm["dischtime"]) - pd.to_datetime(adm["admittime"])).dt.total_seconds() / 86400

# Chart style: thin marks, recessive grid, one hue for single-series charts
BLUE, GREY, INK = "#2a78d6", "#c9c8c2", "#0b0b0b"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": GREY, "axes.grid": True, "grid.color": "#ebeae6",
                     "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 10})

los = master["los_days"]

# 1. LOS summary
print("LOS (days): mean %.2f | median %.2f | std %.2f | skew %.2f"
      % (los.mean(), los.median(), los.std(), los.skew()))
print("log1p(LOS) skew: %.2f" % np.log1p(los).skew())
print("quantiles:", los.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).round(1).to_dict())
print("stays < 0.25 day:", int((los < 0.25).sum()), "| stays > 30 days:", int((los > 30).sum()))
print("Reference only: MAE of always predicting the median = %.2f days "
      "(the real baseline is computed in CV, Step 5)" % np.abs(los - los.median()).mean())

# Analysis only (hospital_expire_flag is never a feature): do deaths cut stays short?
short = adm["los_days"] < 0.25
print("short stays ending in in-hospital death:", int(adm.loc[short, "hospital_expire_flag"].sum()),
      "of", int(short.sum()))
print("median LOS by in-hospital death:", adm.groupby("hospital_expire_flag")["los_days"].median().round(2).to_dict())

# 2. LOS distribution, raw and log scale (two panels, one axis each)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(los, bins=30, color=BLUE, edgecolor="white", linewidth=0.8)
ax[0].axvline(los.median(), color=INK, lw=1)
ax[0].text(los.median(), ax[0].get_ylim()[1] * 0.92, f"  median {los.median():.1f}", color=INK, va="top")
ax[0].set(title="Length of stay", xlabel="days", ylabel="admissions")
ax[1].hist(np.log1p(los), bins=30, color=BLUE, edgecolor="white", linewidth=0.8)
ax[1].set(title="Length of stay, log scale", xlabel="log(1 + days)")
plt.tight_layout(); plt.show()

# 3. O2 label groups
grp = np.select([(master["diabetes"] == 1) & (master["heart_disease"] == 1),
                 master["diabetes"] == 1, master["heart_disease"] == 1],
                ["Both", "Diabetes only", "Heart only"], "Neither")
order = ["Neither", "Diabetes only", "Heart only", "Both"]
counts = pd.Series(grp).value_counts().reindex(order)
assert counts.sum() == 129
fig, ax = plt.subplots(figsize=(5.5, 3))
bars = ax.bar(order, counts.values, color=BLUE, width=0.55)
ax.bar_label(bars, padding=2)
ax.set(title="O2 label groups", ylabel="admissions")
ax.grid(axis="x", visible=False)
plt.show()
print(counts.to_dict(), "| Step 1 implies neither 52, diabetes-only 25, heart-only 29, both 23")

# 4. The patient with the most admissions vs everyone else
top = master["subject_id"].value_counts().idxmax()
is_top = master["subject_id"] == top
label = np.where(is_top, f"top patient ({is_top.sum()} admissions)", "all others")
display(master.assign(group=label).groupby("group").agg(
    admissions=("hadm_id", "size"), los_median=("los_days", "median"), los_mean=("los_days", "mean"),
    diabetes_rate=("diabetes", "mean"), heart_rate=("heart_disease", "mean"),
    age_median=("age", "median")).round(2))

### 4a findings: targets
- **LOS (O1):** mean 9.33, median 6.63, skew 6.15; log(1 + days) skew 0.14, so model on the log scale and report error in days. Quantiles: 90% 18.2, 95% 26.9, 99% 38.7. 5 stays > 30 days; one stay of ~124 days is isolated, so MAE is the headline metric and results are also reported without it.
- **Deaths:** 3 stays < 0.25 day, all in-hospital deaths. Median LOS 4.8 d (died) vs 6.8 d (survived), so LOS for deaths is time to death. The death flag is a leakage column, so this stays as noise.
- **Reference only:** always predicting the median gives MAE 5.95 days; the real baseline is computed inside CV.
- **O2 labels:** neither 52, diabetes only 25, heart only 29, both 23 (matches Step 1).
- **Top patient (15 admissions):** diabetes in ~14 of 15 admissions, no heart disease, median age 65 (others 76), median LOS 4.1 (others 6.8). They supply 14 of 48 diabetes-positive admissions (29%) but are 1 of 31 diabetes-positive patients.
- **Decision:** weight training rows by 1 / admissions-per-patient and report O2 with and without this patient.

In [ ]:
labs = master.filter(like="lab_")
short = lambda c: c.split("_", 2)[2]                     # lab_50868_anion_gap -> anion_gap

# 1. Categorical features and age
for c in ["gender", "admission_type_grp", "insurance_grp", "ethnicity_grp",
          "marital_grp", "admission_location_grp", "first_careunit_24h"]:
    print(f"{c}:", master[c].value_counts().to_dict())
print("\nage:", master["age"].describe().round(1).to_dict(), "| shifted:", int(master["age_was_shifted"].sum()))

# 2. Lab missingness (imputed later, inside the Pipeline)
miss = (labs.isna().mean() * 100).round(1).sort_values(ascending=False)
print("\nlabs with most missing (%):", {short(k): v for k, v in miss.head(6).items()})

# 3. Lab pairs that say nearly the same thing (|Spearman| >= 0.85), to prune later
corr = labs.corr(method="spearman")
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().rename("rho").reset_index()
pairs = pairs[pairs["rho"].abs() >= 0.85].sort_values("rho", key=abs, ascending=False)
print("\nhighly correlated lab pairs:")
for _, r in pairs.iterrows():
    print(f"  {short(r['level_0'])} ~ {short(r['level_1'])}: {r['rho']:.2f}")

# 4. Prescription counts: do the two features overlap?
rx_cols = ["n_orders_24h", "n_unique_drugs_24h"]
print("\nrx counts correlation:", round(master[rx_cols].corr(method="spearman").iloc[0, 1], 2))

# 5. Lab correlation heatmap (diverging, neutral at 0)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels([short(c) for c in corr.columns], rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels([short(c) for c in corr.index], fontsize=7)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman correlation")
plt.title("Lab correlations (first 24 h)"); plt.tight_layout(); plt.show()

### 4b findings: features
- **Age:** mean 70.3, std 16.4, median 73, max 90 (9 capped). Gender: 70 M / 59 F admissions.
- **Missing labs:** worst PTT 10.9%, INR and PT 10.1%, phosphate and calcium 6.2%; imputed later inside the Pipeline.
- **Correlated labs (|rho| >= 0.85):** INR~PT 0.99, hematocrit~hemoglobin 0.94, MCH~MCV 0.88, hematocrit~RBC 0.87. Drop PT, hematocrit, MCH (20 labs remain). Chosen without using the target.
- **Prescription counts:** n_orders_24h ~ n_unique_drugs_24h = 0.98; keep n_unique_drugs_24h only.
- **ICU unit groups:** SICU + TSICU + CSRU -> SURGICAL_ICU (33); MICU 65, CCU 14, NONE_BY_24H 17.
- **Feature set v1:** age, age_was_shifted, gender, 20 labs, n_unique_drugs_24h, admission type, admission location, insurance, icu_by_24h, ICU unit group. Ethnicity and marital status held out (tiny groups, not in the guideline's variable list); tested later as a comparison.

In [ ]:
master["age_band"] = pd.cut(master["age"], bins=[0, 64, 79, 90], labels=["<65", "65-79", "80+"])

def group_summary(by):
    return master.groupby(by, observed=True).agg(
        admissions=("hadm_id", "size"), patients=("subject_id", "nunique"),
        los_median=("los_days", "median"), diabetes_rate=("diabetes", "mean"),
        heart_rate=("heart_disease", "mean")).round(2)

display(group_summary("gender"))
display(group_summary("age_band"))
display(group_summary(["gender", "age_band"]))

### 4c findings: group differences (descriptive)
- **By age band (admissions):** <65: 36, 65-79: 46, 80+: 47. Heart-disease rate 0.11 / 0.39 / 0.64; diabetes rate flat (0.36-0.39); median LOS 7.7 / 6.8 / 5.1 days.
- **By gender:** F 59 adm (55 patients), M 70 (45). Heart rate F 0.49 vs M 0.33; diabetes F 0.32 vs M 0.41. Women are older (31 of 59 F admissions are 80+ vs 16 of 70 M), so part of the gap is age.
- **Implications:** add an age-only baseline for the heart label; report model metrics by gender and age band with spread; treat cells with < 20 admissions (e.g. female <65, n = 9) as too small to interpret.

## 5. Evaluation plan (stated before any model is tuned)

| | Task | Metrics | Baselines | Models |
| --- | --- | --- | --- | --- |
| O1 | Regression on los_days (modelled on log(1 + days), error reported in days) | MAE (primary), RMSE, R² | Training-fold median | Ridge, Random Forest |
| O2 | Two binary classifiers: diabetes, heart disease | F1, ROC-AUC, accuracy | Majority class; age-only logistic regression | Logistic regression, XGBoost, MLP |
| O3 | Clustering on the feature set | Silhouette, Calinski-Harabasz, clinical interpretation | none | k-means / HDBSCAN |

- **Split:** 5-fold StratifiedGroupKFold on subject_id (GroupKFold plus label balance), seed 42. No patient appears in both train and test.
- **Preprocessing inside each fold:** median imputation, scaling and one-hot encoding fitted on training folds only, in a Pipeline.
- **Sample weights (O2):** 1 / admissions-per-patient; results also reported without the 15-admission patient.
- **Reporting:** cross-validated mean with spread, never a single score. With 129 admissions, say so in every result.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
master["label_group"] = np.select(
    [(master["diabetes"] == 1) & (master["heart_disease"] == 1),
     master["diabetes"] == 1, master["heart_disease"] == 1],
    ["both", "diabetes_only", "heart_only"], "neither")
top = master["subject_id"].value_counts().idxmax()          # patient with the most admissions

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
for k, (tr, te) in enumerate(sgkf.split(master, master["label_group"], groups=master["subject_id"])):
    t = master.iloc[te]
    assert set(master.iloc[tr]["subject_id"]).isdisjoint(t["subject_id"]), "patient in both train and test"
    rows.append({"fold": k, "test_admissions": len(t), "test_patients": t["subject_id"].nunique(),
                 "diabetes_pos": int(t["diabetes"].sum()), "heart_pos": int(t["heart_disease"].sum()),
                 "los_median": round(t["los_days"].median(), 2),
                 "has_top_patient": bool((t["subject_id"] == top).any())})
display(pd.DataFrame(rows))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

# --- Feature spec (decisions from 4b) ---
master["icu_unit_grp"] = master["first_careunit_24h"].replace(
    {"SICU": "SURGICAL_ICU", "TSICU": "SURGICAL_ICU", "CSRU": "SURGICAL_ICU"})
DROP_LABS = ["lab_51274_pt", "lab_51221_hematocrit", "lab_51248_mch"]       # near-duplicates (|rho| >= 0.85)
LAB_COLS = [c for c in master.columns if c.startswith("lab_") and c not in DROP_LABS]
NUM = ["age"] + LAB_COLS + ["n_unique_drugs_24h"]
BIN = ["age_was_shifted", "icu_by_24h"]
CAT = ["gender", "admission_type_grp", "admission_location_grp", "insurance_grp", "icu_unit_grp"]
FEATURES = NUM + BIN + CAT
assert len(LAB_COLS) == 20, len(LAB_COLS)
assert not {"los_days", "diabetes", "heart_disease", "subject_id", "hadm_id", "label_group"} & set(FEATURES)
assert not FORBIDDEN & set(FEATURES), "leakage column in features"

# --- Preprocessing: fitted on training folds only (used by the models in the next step) ---
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM),
    ("bin", "passthrough", BIN),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])

# --- Same folds for every model; patient weights for the weighted score ---
y = master["los_days"]
w = 1 / master.groupby("subject_id")["hadm_id"].transform("size")       # each patient counts once
cv = list(sgkf.split(master, master["label_group"], groups=master["subject_id"]))

def cv_mae(model, X):
    """MAE in days per fold: plain, and patient-weighted."""
    plain, weighted = [], []
    for tr, te in cv:
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict(X.iloc[te])
        plain.append(mean_absolute_error(y.iloc[te], p))
        weighted.append(mean_absolute_error(y.iloc[te], p, sample_weight=w.iloc[te]))
    return np.array(plain), np.array(weighted)

# --- O1 baseline: always predict the training-fold median ---
X_all = master[FEATURES]
plain, weighted = cv_mae(DummyRegressor(strategy="median"), X_all)
print("Baseline (median) MAE, days, per fold plain:", plain.round(2))
print("  plain    : %.2f +/- %.2f" % (plain.mean(), plain.std(ddof=1)))
print("  weighted : %.2f +/- %.2f" % (weighted.mean(), weighted.std(ddof=1)))
print("features:", len(FEATURES), "| numeric", len(NUM), "| binary", len(BIN), "| categorical", len(CAT))

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

def make_model(est):
    """Preprocessing + model, trained on log(1 + days); predictions come back in days."""
    pipe = Pipeline([("prep", preprocess), ("model", est)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

res = {"median baseline": cv_mae(DummyRegressor(strategy="median"), X_all)}
res["Ridge (alpha=1)"] = cv_mae(make_model(Ridge(alpha=1.0)), X_all)
res["Random Forest"] = cv_mae(make_model(
    RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1)), X_all)

# --- Summary: mean +/- spread, and how many folds beat the baseline ---
base_plain = res["median baseline"][0]
rows = []
for name, (p, wt) in res.items():
    rows.append({"model": name,
                 "MAE plain": f"{p.mean():.2f} +/- {p.std(ddof=1):.2f}",
                 "MAE patient-weighted": f"{wt.mean():.2f} +/- {wt.std(ddof=1):.2f}",
                 "folds beating baseline": "-" if name == "median baseline" else f"{int((p < base_plain).sum())}/5"})
display(pd.DataFrame(rows))
for name, (p, wt) in res.items():
    print(f"{name:18s} per-fold plain MAE:", p.round(2))

# --- Which fold holds the longest stay and the top patient? ---
fold_of = np.full(len(master), -1)
for k, (tr, te) in enumerate(cv):
    fold_of[te] = k
print("\nfold with the longest stay:", int(fold_of[int(np.argmax(y.values))]),
      "| fold with the top patient:", int(fold_of[(master["subject_id"] == top).values][0]))

In [ ]:
tr, te = cv[1]                                              # the fold where Ridge was worst
m = make_model(Ridge(alpha=1.0)).fit(X_all.iloc[tr], y.iloc[tr])
p = m.predict(X_all.iloc[te])

err = pd.DataFrame({"actual": y.iloc[te].values, "pred": p})
err["abs_err"] = (err["actual"] - err["pred"]).abs()
mu, sd = X_all.iloc[tr][NUM].mean(), X_all.iloc[tr][NUM].std()
err["max_abs_z"] = ((X_all.iloc[te][NUM] - mu) / sd).abs().max(axis=1).values   # most extreme numeric feature
display(err.sort_values("abs_err", ascending=False).head(6).round(1))
print("test rows with a feature more than 4 SD from the training mean:", int((err["max_abs_z"] > 4).sum()),
      "of", len(err))

In [ ]:
from sklearn.model_selection import GridSearchCV, GroupKFold

groups = master["subject_id"]

def cv_mae_nested(search, X):
    """Outer folds score the model; the alpha is chosen only inside each training fold."""
    plain, weighted, chosen = [], [], []
    for tr, te in cv:
        search.fit(X.iloc[tr], y.iloc[tr], groups=groups.iloc[tr])     # inner grouped CV picks alpha
        p = search.predict(X.iloc[te])
        plain.append(mean_absolute_error(y.iloc[te], p))
        weighted.append(mean_absolute_error(y.iloc[te], p, sample_weight=w.iloc[te]))
        chosen.append(search.best_params_["regressor__model__alpha"])
    return np.array(plain), np.array(weighted), chosen

ridge_search = GridSearchCV(
    make_model(Ridge()),
    param_grid={"regressor__model__alpha": [1, 10, 100, 1000, 10000]},
    scoring="neg_mean_absolute_error",                                 # MAE in days
    cv=GroupKFold(n_splits=3))

p, wt, chosen = cv_mae_nested(ridge_search, X_all)
base_plain = res["median baseline"][0]
print("Ridge tuned (nested): plain %.2f +/- %.2f | weighted %.2f +/- %.2f | folds beating baseline: %d/5"
      % (p.mean(), p.std(ddof=1), wt.mean(), wt.std(ddof=1), int((p < base_plain).sum())))
print("per-fold plain MAE:", p.round(2), "| baseline:", base_plain.round(2))
print("alpha chosen per fold:", chosen)

### 5e: Ridge tuned with nested CV
- Alpha chosen by inner grouped 3-fold CV within each training fold from [1, 10, 100, 1000, 10000]: [1000, 1000, 10000, 1000, 100].
- MAE plain 6.01 +/- 2.37, patient-weighted 6.22 +/- 2.28 (baseline 6.05 / 6.28). Per-fold gap to baseline is about +0.04 days on average (SE ~0.07): not distinguishable from zero.
- Large chosen alphas mean the model ignores most features, so the first-24h features carry little usable signal for LOS here. **O1 conclusion: no meaningful improvement over the median baseline.**

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

def oof_predict(model, X, tuned=False):
    """Out-of-fold predictions: each admission is predicted by a model trained without its patient."""
    pred = np.zeros(len(X))
    for tr, te in cv:
        if tuned:
            model.fit(X.iloc[tr], y.iloc[tr], groups=groups.iloc[tr])
        else:
            model.fit(X.iloc[tr], y.iloc[tr])
        pred[te] = model.predict(X.iloc[te])
    return pred

preds = {
    "median baseline": oof_predict(DummyRegressor(strategy="median"), X_all),
    "Ridge (tuned)": oof_predict(ridge_search, X_all, tuned=True),
    "Random Forest": oof_predict(make_model(RandomForestRegressor(
        n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1)), X_all),
}

longest = int(np.argmax(y.values))
rows = []
for scope, mask in [("all 129", np.ones(len(y), dtype=bool)),
                    ("without the longest stay", np.arange(len(y)) != longest)]:
    for name, pred in preds.items():
        yt, yp = y.values[mask], pred[mask]
        rows.append({"scope": scope, "model": name,
                     "MAE": round(mean_absolute_error(yt, yp), 2),
                     "RMSE": round(float(np.sqrt(mean_squared_error(yt, yp))), 2),
                     "R2": round(r2_score(yt, yp), 3)})
display(pd.DataFrame(rows))

### 5f: O1 final (out-of-fold predictions, pooled)
| Scope | Model | MAE | RMSE | R2 |
| --- | --- | --- | --- | --- |
| all 129 | median baseline | 6.03 | 13.04 | -0.058 |
| all 129 | Ridge (tuned) | 6.00 | 13.05 | -0.061 |
| all 129 | Random Forest | 6.04 | 12.98 | -0.049 |
| without longest stay | median baseline | 5.16 | 7.93 | -0.080 |
| without longest stay | Ridge (tuned) | 5.12 | 7.95 | -0.084 |
| without longest stay | Random Forest | 5.17 | 7.86 | -0.059 |

**Conclusion:** no model beats the median baseline in any meaningful way; R2 is slightly negative for all. The first-24h features on 129 admissions carry essentially no usable signal for length of stay. One 124-day stay accounts for much of RMSE (13.0 -> 7.9 without it).
**To raise with Bala:** whether vitals (CHARTEVENTS, currently out of scope) or a different framing of O1 should be tried. This is his decision, not a change we make on our own.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def cv_classify(make, X, target):
    """5 grouped folds; patient-weighted training; metrics plain and patient-weighted."""
    yt = master[target].values
    out = {"f1": [], "auc": [], "acc": [], "f1_w": [], "auc_w": []}
    for tr, te in cv:
        model = make()
        model.fit(X.iloc[tr], yt[tr], model__sample_weight=w.values[tr])
        proba = model.predict_proba(X.iloc[te])[:, 1]
        pred = (proba >= 0.5).astype(int)
        sw = w.values[te]
        out["f1"].append(f1_score(yt[te], pred, zero_division=0))
        out["auc"].append(roc_auc_score(yt[te], proba))
        out["acc"].append(accuracy_score(yt[te], pred))
        out["f1_w"].append(f1_score(yt[te], pred, sample_weight=sw, zero_division=0))
        out["auc_w"].append(roc_auc_score(yt[te], proba, sample_weight=sw))
    return out

def fmt(a):
    return f"{np.mean(a):.2f} +/- {np.std(a, ddof=1):.2f}"

models = {
    "majority class":      (lambda: Pipeline([("model", DummyClassifier(strategy="prior"))]), X_all),
    "age-only logistic":   (lambda: Pipeline([("scale", StandardScaler()),
                                              ("model", LogisticRegression(max_iter=1000))]), master[["age"]]),
    "logistic (all features)": (lambda: Pipeline([("prep", clone(preprocess)),
                                                  ("model", LogisticRegression(max_iter=1000))]), X_all),
}

for target in ["diabetes", "heart_disease"]:
    rows = []
    for name, (make, X) in models.items():
        r = cv_classify(make, X, target)
        rows.append({"model": name, "F1": fmt(r["f1"]), "ROC-AUC": fmt(r["auc"]), "accuracy": fmt(r["acc"]),
                     "F1 (patient-wtd)": fmt(r["f1_w"]), "AUC (patient-wtd)": fmt(r["auc_w"])})
    print(f"\n=== {target} (positives: {int(master[target].sum())} of {len(master)}) ===")
    display(pd.DataFrame(rows))

In [ ]:
def cv_c(C, X, target, drop_subject=None):
    """Logistic with L2 strength C; optionally drop one patient from train and test."""
    yt = master[target].values
    keep = np.ones(len(master), bool) if drop_subject is None else (master.subject_id.values != drop_subject)
    aucs = []
    for tr, te in cv:
        tr, te = tr[keep[tr]], te[keep[te]]
        if len(np.unique(yt[te])) < 2:      # AUC needs both classes in the test fold
            continue
        m = Pipeline([("prep", clone(preprocess)),
                      ("model", LogisticRegression(C=C, max_iter=1000))])
        m.fit(X.iloc[tr], yt[tr], model__sample_weight=w.values[tr])
        aucs.append(roc_auc_score(yt[te], m.predict_proba(X.iloc[te])[:, 1]))
    return aucs

top_diab = master.loc[master.diabetes == 1, "subject_id"].value_counts().index[0]
rows = []
for C in [0.01, 0.03, 0.1, 0.3, 1.0]:
    rows.append({"C": C,
                 "heart AUC": fmt(cv_c(C, X_all, "heart_disease")),
                 "diabetes AUC": fmt(cv_c(C, X_all, "diabetes")),
                 "diabetes AUC (no top patient)": fmt(cv_c(C, X_all, "diabetes", drop_subject=top_diab))})
display(pd.DataFrame(rows))

In [ ]:
from xgboost import XGBClassifier

def cv_auc_model(make, X, target):
    yt = master[target].values
    aucs = []
    for tr, te in cv:
        if len(np.unique(yt[te])) < 2:
            continue
        m = make()
        m.fit(X.iloc[tr], yt[tr], model__sample_weight=w.values[tr])
        aucs.append(roc_auc_score(yt[te], m.predict_proba(X.iloc[te])[:, 1]))
    return aucs

make_xgb = lambda: Pipeline([("prep", clone(preprocess)),
                             ("model", XGBClassifier(max_depth=2, n_estimators=100, learning_rate=0.05,
                                                     subsample=0.8, colsample_bytree=0.8,
                                                     eval_metric="logloss", random_state=SEED))])
rows = [{"label": t, "XGBoost AUC": fmt(cv_auc_model(make_xgb, X_all, t))}
        for t in ["diabetes", "heart_disease"]]
display(pd.DataFrame(rows))

### 6: O2 final (grouped 5-fold CV, patient-weighted training, ROC-AUC mean +/- SD)
| Label | Majority | Age-only | Logistic C=0.03 | XGBoost (depth 2) |
| --- | --- | --- | --- | --- |
| diabetes (48/129) | 0.50 | 0.57 +/- 0.12 | 0.66 +/- 0.11 | 0.59 +/- 0.17 |
| heart disease (52/129) | 0.50 | 0.77 +/- 0.03 | 0.69 +/- 0.07 | 0.67 +/- 0.13 |

- Diabetes without its top patient: logistic AUC 0.71 +/- 0.12 (signal not driven by one patient).
- Heart: no model beats age alone, so the other features add nothing.
- C was picked by looking at these same folds (slightly optimistic). The MLP was not run: too few rows for it to be reliable.
- Near-label features: glucose (diabetes), CCU stay (heart).
- To raise with Bala: whether vitals (CHARTEVENTS) or more admissions are available; whether the MLP is still wanted.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Z = clone(preprocess).fit_transform(X_all)     # impute + scale + one-hot, same as models
rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Z)
    rows.append({"k": k, "silhouette": round(silhouette_score(Z, km.labels_), 3),
                 "smallest cluster": int(np.bincount(km.labels_).min())})
display(pd.DataFrame(rows))

In [ ]:
km2 = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit(Z)
prof = master.assign(cluster=km2.labels_).groupby("cluster").agg(
    n=("hadm_id", "size"), age_median=("age", "median"),
    los_median=("los_days", "median"), icu_rate=("icu_by_24h", "mean"),
    diabetes=("diabetes", "mean"), heart=("heart_disease", "mean")).round(2)
display(prof)

In [ ]:
cl1 = master.assign(cluster=km2.labels_).query("cluster == 1")
print("cluster-1 admissions by patient:")
print(cl1["subject_id"].value_counts().head(5))
print("top diabetic patient:", top_diab)

In [ ]:
keep = master["subject_id"] != 41976
Z2 = clone(preprocess).fit_transform(X_all[keep])   # refit scaling without this patient
rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Z2)
    rows.append({"k": k, "silhouette": round(silhouette_score(Z2, km.labels_), 3),
                 "smallest cluster": int(np.bincount(km.labels_).min())})
print("admissions kept:", int(keep.sum()))
display(pd.DataFrame(rows))

### 7: O3 clustering (k-means on the 29 first-24h features, scaled)
| k | silhouette (all 129) | silhouette (without subject 41976) |
| --- | --- | --- |
| 2 | 0.142 | 0.078 |
| 3 | 0.101 | 0.083 |
| 4 | 0.105 | 0.067 |
| 5 | 0.079 | 0.060 |
| 6 | 0.070 | 0.069 |

**Conclusion:** no cluster structure (all silhouettes below 0.25). The only apparent split (k=2 on all 129) was one patient's 15 repeat admissions (subject 41976) plus 2 others; removing that patient leaves silhouettes of 0.06–0.08. HDBSCAN was not run; to raise with Bala.

## HealthAI Suite Phase 1: summary for Bala
**What I did:** quality checks and cleaning on 8 MIMIC-III demo tables (129 admissions, 100 patients), one master table (one row per admission, first-24h features only, no leakage columns), EDA, then modelling with grouped 5-fold CV (by patient).

**Findings**
- O1 (LOS regression): no model beats the median baseline (MAE about 6.0 days; R2 slightly negative). One 124-day stay dominates RMSE.
- O2 (classification, ROC-AUC): heart disease is predicted by age alone (0.77 +/- 0.03). Logistic on all features (0.69) and XGBoost (0.67) do not beat it. Diabetes is weak (about 0.66 +/- 0.11, near-label glucose), and it holds without the top patient.
- O3 (clustering): no structure (silhouette below 0.15). The one apparent cluster was a single patient's repeat admissions.

**Assumptions / limits:** 129 admissions is too small for a reliable signal; the demo set is selected from patients who died; C was chosen on the same folds (slightly optimistic); MLP and HDBSCAN were not run.

**Questions for you:** (1) Add vitals (CHARTEVENTS) for O1/O2? (2) Is a negative O1 result acceptable for the demo set, or should I reframe it? (3) Do you want the MLP and HDBSCAN run anyway?